In [3]:
## =====================================================
## STEP 9 — Predict VEI for Missing Rows
## Dataset: Merged_Volcano_Eruptions.xlsx
## =====================================================

import pandas as pd
import numpy as np
import re
import joblib

print("\n===== STEP 9 — Starting VEI Prediction Pipeline =====\n")

# ---------------------------------------------------------
# 1. LOAD MERGED DATASET
# ---------------------------------------------------------
df = pd.read_excel("Merged_Volcano_Eruptions.xlsx")

# Clean column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Merged dataset loaded:", df.shape)

# ---------------------------------------------------------
# 2. CLEAN VEI COLUMN + IDENTIFY MISSING ROWS
# ---------------------------------------------------------
df["VEI"] = pd.to_numeric(df["VEI"], errors="coerce")

missing_before = df["VEI"].isna().sum()
print("Missing VEI rows before engineering:", missing_before)

if missing_before == 0:
    print("No missing VEI values to predict. Exiting.")
    raise SystemExit


# ---------------------------------------------------------
# 3. FEATURE ENGINEERING (Same as Step 3E)
# ---------------------------------------------------------

# Extract numeric year from LAST KNOWN ERUPTION
def extract_year(text):
    if isinstance(text, str):
        m = re.search(r"(\d{3,4})", text)
        if m:
            return int(m.group(1))
    return np.nan

df["Last_Eruption_Year"] = df["Last Known Eruption"].apply(extract_year)
df["Last_Eruption_Year"] = df["Last_Eruption_Year"].fillna(0)

# Dormancy
df["Dormancy"] = df["Start Year"] - df["Last_Eruption_Year"]
df.loc[df["Dormancy"] < 0, "Dormancy"] = 0

# Log-dormancy
df["Dormancy_Log"] = np.log1p(df["Dormancy"])

# Dormancy class
def dormancy_class(x):
    if x < 20: return "Short"
    elif x < 100: return "Medium"
    else: return "Long"

df["Dormancy_Class"] = df["Dormancy"].apply(dormancy_class)

# Morphology grouping
def morphology_group(v):
    if isinstance(v, str):
        U = v.upper()
        if "CALDERA" in U or "MAAR" in U or "TUFF" in U:
            return "Explosive"
        if "SHIELD" in U or "FISSURE" in U:
            return "Effusive"
    return "Mixed"

df["Morphology_Group"] = df["Primary Volcano Type"].apply(morphology_group)

# Tectonic grouping
def tect_group(t):
    if isinstance(t, str):
        T = t.upper()
        if "SUBDUCTION" in T: return "Subduction"
        if "RIFT" in T: return "Rift"
        if "INTRAPLATE" in T: return "Hotspot"
    return "Other"

df["Tectonic_Group"] = df["Tectonic Setting"].apply(tect_group)

# Interaction features
df["Dormancy_Morphology"] = df["Dormancy_Class"] + "_" + df["Morphology_Group"]
df["Type_Tectonic"] = df["Primary Volcano Type"] + "_" + df["Tectonic Setting"]
df["Region_Type"] = df["Volcanic Region"] + "_" + df["Primary Volcano Type"]
df["Type_Landform"] = df["Primary Volcano Type"] + "_" + df["Volcano Landform"]
df["Region_Tectonic"] = df["Volcanic Region Group"] + "_" + df["Tectonic_Group"]

print("Feature engineering complete.")


# ---------------------------------------------------------
# 4. RECREATE df_missing AFTER ALL ENGINEERING
# ---------------------------------------------------------
df_missing = df[df["VEI"].isna()].copy()
print("Missing VEI rows AFTER engineering:", df_missing.shape[0])

# Prepare X for model (drop VEI column)
X_missing = df_missing.drop(columns=["VEI"], errors="ignore")


# ---------------------------------------------------------
# 5. LOAD FINAL TRAINED MODEL
# ---------------------------------------------------------
print("\nLoading model: Final_XGBoost_Model.pkl")
model = joblib.load("Final_XGBoost_Model.pkl")


# ---------------------------------------------------------
# 6. PREDICT VEI CLASS (0=Low, 1=Moderate, 2=High)
# ---------------------------------------------------------
y_pred = model.predict(X_missing)

inv_map = {0: "Low", 1: "Moderate", 2: "High"}
df_missing["Predicted_VEI_Class"] = [inv_map[i] for i in y_pred]


# ---------------------------------------------------------
# 7. SAVE RESULTS
# ---------------------------------------------------------
output_file = "Predicted_Missing_VEI.xlsx"
df_missing.to_excel(output_file, index=False)

print("\n===== PREDICTION COMPLETE =====")
print("Saved predicted VEI classes to:", output_file)

print("\nSample Predictions:")
print(df_missing[[
    "Volcano Name_x",
    "Start Year",
    "Primary Volcano Type",
    "Tectonic Setting",
    "Dormancy",
    "Predicted_VEI_Class"
]].head(10))



===== STEP 9 — Starting VEI Prediction Pipeline =====

Merged dataset loaded: (9902, 37)
Missing VEI rows before engineering: 2203
Feature engineering complete.
Missing VEI rows AFTER engineering: 2203

Loading model: Final_XGBoost_Model.pkl


c:\Users\SHIVAM\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [1, 2, 3, 4, 5, 7, 9, 14, 15, 16, 17, 18, 19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(



===== PREDICTION COMPLETE =====
Saved predicted VEI classes to: Predicted_Missing_VEI.xlsx

Sample Predictions:
           Volcano Name_x  Start Year Primary Volcano Type  \
0                Karymsky        2025        Stratovolcano   
1                 Bulusan        2025    Stratovolcano(es)   
3            Klyuchevskoy        2025        Stratovolcano   
4               Reykjanes        2025          Crater rows   
5                  Ulawun        2025        Stratovolcano   
6                   Raung        2025        Stratovolcano   
7   Atka Volcanic Complex        2025    Stratovolcano(es)   
8                  Purace        2025    Stratovolcano(es)   
9               Lewotolok        2025        Stratovolcano   
10                 Telica        2025    Stratovolcano(es)   

                                   Tectonic Setting  Dormancy  \
0      Subduction zone / Continental crust (>25 km)       0.0   
1      Subduction zone / Continental crust (>25 km)       0.0   
3      Su